## inference code for whisper small

In [1]:
from safetensors import safe_open
import json

file_location = "model.safetensors"

with safe_open(file_location, framework="pt", device='cpu') as f: 
    model_structure = {
        layer: list(f.get_tensor(layer).shape)
        for layer in f.keys()
    }

    with open("model_safetensors.json", "w") as json_file:
        json.dump(model_structure, json_file, indent=4)
    
    for layer, shape in model_structure.items():
        if 'model.decoder.layers' in layer:
            print(f"Name: {layer}, shape: {shape}")

Name: model.decoder.layers.0.encoder_attn.k_proj.weight, shape: [768, 768]
Name: model.decoder.layers.0.encoder_attn.out_proj.bias, shape: [768]
Name: model.decoder.layers.0.encoder_attn.out_proj.weight, shape: [768, 768]
Name: model.decoder.layers.0.encoder_attn.q_proj.bias, shape: [768]
Name: model.decoder.layers.0.encoder_attn.q_proj.weight, shape: [768, 768]
Name: model.decoder.layers.0.encoder_attn.v_proj.bias, shape: [768]
Name: model.decoder.layers.0.encoder_attn.v_proj.weight, shape: [768, 768]
Name: model.decoder.layers.0.encoder_attn_layer_norm.bias, shape: [768]
Name: model.decoder.layers.0.encoder_attn_layer_norm.weight, shape: [768]
Name: model.decoder.layers.0.fc1.bias, shape: [3072]
Name: model.decoder.layers.0.fc1.weight, shape: [3072, 768]
Name: model.decoder.layers.0.fc2.bias, shape: [768]
Name: model.decoder.layers.0.fc2.weight, shape: [768, 3072]
Name: model.decoder.layers.0.final_layer_norm.bias, shape: [768]
Name: model.decoder.layers.0.final_layer_norm.weight, sh

In [2]:
import torch
from safetensors import safe_open

# In Whisper, k_proj has no bias across all three attention modules.
# (encoder self-attention, decoder self-attention, and decoder cross-attention).


class WhisperEncoderAttention(torch.nn.Module):
    def __init__(self, config: dict, layer_no: int, safetensor_files: list):
        super().__init__()

        self.d_model = config["d_model"]
        self.num_head = config["encoder_attention_heads"]

        self.files = safetensor_files

        assert self.d_model % self.num_head == 0
        self.head_dim = self.d_model // self.num_head

        self.q_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.k_proj = torch.nn.Linear(self.d_model, self.d_model, bias=False)
        self.v_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.out_proj = torch.nn.Linear(self.d_model, self.d_model)

        # loading pretrained weights
        self.q_proj.weight.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.self_attn.q_proj.weight")
        )
        self.q_proj.bias.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.self_attn.q_proj.bias")
        )
        self.k_proj.weight.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.self_attn.k_proj.weight")
        )
        self.v_proj.weight.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.self_attn.v_proj.weight")
        )
        self.v_proj.bias.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.self_attn.v_proj.bias")
        )
        self.out_proj.weight.data.copy_(
            self.get_tensor(
                f"model.encoder.layers.{layer_no}.self_attn.out_proj.weight"
            )
        )
        self.out_proj.bias.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.self_attn.out_proj.bias")
        )

    @torch.no_grad()
    def forward(self, x):
        B, S, D = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, S, self.num_head, self.head_dim).transpose(1, 2)
        k = k.view(B, S, self.num_head, self.head_dim).transpose(1, 2)
        v = v.view(B, S, self.num_head, self.head_dim).transpose(1, 2)

        attn_score = torch.einsum("bhqd, bhkd -> bhqk", q, k) * (self.head_dim**-0.5)
        wei = torch.nn.functional.softmax(attn_score, dim=-1)

        v_rep = wei @ v
        out = v_rep.transpose(1, 2).contiguous()
        out = out.view(B, S, self.d_model)

        return self.out_proj(out)

    def get_tensor(self, name):
        for f in self.files:
            if name in f.keys():
                return f.get_tensor(name)

In [ ]:
class WhisperEncoderFeedForwardNeuralNetwork(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, safetensor_files: list):
        super().__init__()

        self.d_model = config["d_model"]
        self.d_ff = config["encoder_ffn_dim"]

        self.files = safetensor_files

        self.fc1 = torch.nn.Linear(in_features=self.d_model, out_features=self.d_ff)
        self.fc2 = torch.nn.Linear(in_features=self.d_ff, out_features=self.d_model)

        # loading pretrained weights
        self.fc1.weight.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.fc1.weight")
        )
        self.fc1.bias.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.fc1.bias")
        )
        self.fc2.weight.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.fc2.weight")
        )
        self.fc2.bias.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.fc2.bias")
        )

    @torch.no_grad()
    def forward(self, x):

        output = self.fc1(x)
        output = torch.nn.functional.gelu(output)

        return self.fc2(output)

    def get_tensor(self, name):
        for f in self.files:
            if name in f.keys():
                return f.get_tensor(name)

In [ ]:
class WhisperEncoderLayer(torch.nn.Module):

    def __init__(self, config: dict, layer_no, safetensor_files: list):
        super().__init__()

        self.files = safetensor_files

        self.encoder_attn = WhisperEncoderAttention(config, layer_no, self.files)
        self.encoder_ffn = WhisperEncoderFeedForwardNeuralNetwork(
            config, layer_no, self.files
        )

        self.attn_layer_norm = torch.nn.LayerNorm(config["d_model"])
        self.attn_layer_norm.weight.data.copy_(
            self.get_tensor(
                f"model.encoder.layers.{layer_no}.self_attn_layer_norm.weight"
            )
        )
        self.attn_layer_norm.bias.data.copy_(
            self.get_tensor(
                f"model.encoder.layers.{layer_no}.self_attn_layer_norm.bias"
            )
        )

        self.final_layer_norm = torch.nn.LayerNorm(config["d_model"])
        self.final_layer_norm.weight.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.final_layer_norm.weight")
        )
        self.final_layer_norm.bias.data.copy_(
            self.get_tensor(f"model.encoder.layers.{layer_no}.final_layer_norm.bias")
        )

    @torch.no_grad()
    def forward(self, x):

        residual = x

        x = self.encoder_attn(x)
        x = self.attn_layer_norm(x)

        x = residual + x

        residual = x

        x = self.encoder_ffn(x)
        x = self.final_layer_norm(x)

        x = residual + x

        return x

    def get_tensor(self, name):
        for f in self.files:
            if name in f.keys():
                return f.get_tensor(name)

In [5]:
class WhisperEncoder(torch.nn.Module):

    def __init__(self, config: dict, safetensor_files: list[str]):
        super().__init__()

        self.model_config = config
        self.files = [
            safe_open(file, framework="pt", device="cpu") for file in safetensor_files
        ]

        self.conv1 = torch.nn.Conv1d(
            in_channels=self.model_config["num_mel_bins"],
            out_channels=self.model_config["d_model"],
            kernel_size=3,
            padding=1,
        )
        self.conv1.weight.data.copy_(self.get_tensor("model.encoder.conv1.weight"))
        self.conv1.bias.data.copy_(self.get_tensor("model.encoder.conv1.bias"))

        self.conv2 = torch.nn.Conv1d(
            in_channels=self.model_config["d_model"],
            out_channels=self.model_config["d_model"],
            kernel_size=3,
            stride=2,
            padding=1,
        )
        self.conv2.weight.data.copy_(self.get_tensor("model.encoder.conv2.weight"))
        self.conv2.bias.data.copy_(self.get_tensor("model.encoder.conv2.bias"))

        self.embed_positions = torch.nn.Embedding(
            num_embeddings=self.model_config["max_source_positions"],
            embedding_dim=self.model_config["d_model"],
        )
        self.embed_positions.weight.data.copy_(
            self.get_tensor("model.encoder.embed_positions.weight")
        )

        self.layers = torch.nn.ModuleList(
            [
                WhisperEncoderLayer(self.model_config, i, self.files)
                for i in range(self.model_config["encoder_layers"])
            ]
        )

        self.layer_norm = torch.nn.LayerNorm(config["d_model"])
        self.layer_norm.weight.data.copy_(
            self.get_tensor("model.encoder.layer_norm.weight")
        )
        self.layer_norm.bias.data.copy_(
            self.get_tensor("model.encoder.layer_norm.bias")
        )

    @torch.no_grad()
    def forward(self, x):

        x = torch.nn.functional.gelu(self.conv1(x))
        x = torch.nn.functional.gelu(self.conv2(x))

        x = x.transpose(-2, -1)

        positions = torch.arange(x.size(1), device=x.device)

        x = x + self.embed_positions(positions)

        for layer in self.layers:
            x = layer(x)

        return self.layer_norm(x)

    def get_tensor(self, name):

        for f in self.files:
            if name in f.keys():
                return f.get_tensor(name)

## Decoder

In [6]:
class WhisperDecoderMaskedAttention(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, safetensor_files: list):
        super().__init__()

        self.d_model = config["d_model"]
        self.num_head = config["decoder_attention_heads"]

        self.files = safetensor_files

        assert self.d_model % self.num_head == 0
        self.head_dim = self.d_model // self.num_head

        self.q_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.k_proj = torch.nn.Linear(self.d_model, self.d_model, bias=False)
        self.v_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.out_proj = torch.nn.Linear(self.d_model, self.d_model)

        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(
                    config["max_target_positions"], config["max_target_positions"]
                ),
                diagonal=1,
            ).bool(),
        )

        # weight's
        self.q_proj.weight.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.self_attn.q_proj.weight")
        )
        self.k_proj.weight.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.self_attn.k_proj.weight")
        )
        self.v_proj.weight.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.self_attn.v_proj.weight")
        )
        self.out_proj.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.self_attn.out_proj.weight"
            )
        )

        # bias
        self.q_proj.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.self_attn.q_proj.bias")
        )
        self.v_proj.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.self_attn.v_proj.bias")
        )
        self.out_proj.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.self_attn.q_proj.bias")
        )

    @torch.no_grad()
    def forward(self, x):

        B, S, D = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, S, self.num_head, self.head_dim).transpose(1, 2)
        k = k.view(B, S, self.num_head, self.head_dim).transpose(1, 2)
        v = v.view(B, S, self.num_head, self.head_dim).transpose(1, 2)

        attn_score = (q @ k.transpose(-2, -1)) * (self.head_dim**-0.5)
        wei = attn_score.masked_fill(self.mask[:S, :S], float("-inf"))
        wei = torch.nn.functional.softmax(wei, dim=-1)

        v_rep = wei @ v
        out = v_rep.transpose(1, 2).contiguous()
        out = out.view(B, S, self.d_model)

        return self.out_proj(out)

    def get_tensor(self, name):

        for f in self.files:
            if name in f.keys():
                return f.get_tensor(name)

In [10]:
class WhisperDecoderCrossAttention(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, safetensor_files: list):
        super().__init__()

        self.d_model = config["d_model"]
        self.num_head = config["decoder_attention_heads"]

        self.files = safetensor_files

        assert self.d_model % self.num_head == 0
        self.head_dim = self.d_model // self.num_head

        self.q_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.k_proj = torch.nn.Linear(self.d_model, self.d_model, bias=False)
        self.v_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.out_proj = torch.nn.Linear(self.d_model, self.d_model)

        # weight's
        self.q_proj.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn.q_proj.weight"
            )
        )
        self.k_proj.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn.k_proj.weight"
            )
        )
        self.v_proj.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn.v_proj.weight"
            )
        )
        self.out_proj.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn.out_proj.weight"
            )
        )

        # bias
        self.q_proj.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.encoder_attn.q_proj.bias")
        )
        self.v_proj.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.encoder_attn.v_proj.bias")
        )
        self.out_proj.bias.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn.out_proj.bias"
            )
        )

    @torch.no_grad()
    def forward(self, decoder_rep, encoder_rep):

        DB, DS, DD = decoder_rep.shape
        EB, ES, ED = encoder_rep.shape

        q = self.q_proj(decoder_rep)
        k = self.k_proj(encoder_rep)
        v = self.out_proj(encoder_rep)

        q = q.view(DB, DS, self.num_head, self.head_dim).transpose(1, 2)
        k = k.view(EB, ES, self.num_head, self.head_dim).transpose(1, 2)
        v = v.view(EB, ES, self.num_head, self.head_dim).transpose(1, 2)

        attn_score = (q @ k.transpose(-2, -1)) * (self.head_dim**-0.5)
        wei = torch.nn.functional.softmax(attn_score, dim=-1)

        v_rep = wei @ v
        out = v_rep.transpose(1, 2).contiguous()
        out = out.view(DB, DS, self.d_model)

        return self.out_proj(out)

    def get_tensor(self, name):

        for file in self.files:
            if name in file.keys():
                return file.get_tensor(name)

In [14]:
class WhisperDecoderFeedForwardNeuralNetwork(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, safetensor_files: list):
        super().__init__()
        self.d_model = config["d_model"]
        self.d_ff = config["decoder_ffn_dim"]

        self.files = safetensor_files

        self.fc1 = torch.nn.Linear(in_features=self.d_model, out_features=self.d_ff)
        self.fc2 = torch.nn.Linear(in_features=self.d_ff, out_features=self.d_model)

        # weight's
        self.fc1.weight.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.fc1.weight")
        )
        self.fc2.weight.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.fc2.weight")
        )

        # bias
        self.fc1.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.fc1.bias")
        )
        self.fc2.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.fc2.bias")
        )

    @torch.no_grad()
    def forward(self, x):

        output = self.fc1(x)
        output = torch.nn.functional.gelu(output)

        return self.fc2(output)

    def get_tensor(self, name):

        for file in self.files:
            if name in file.keys():
                return file.get_tensor(name)

In [16]:
class WhisperDecoderLayer(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, safetensor_files: list):
        super().__init__()

        self.d_model = config["d_model"]
        self.files = safetensor_files

        self.decoder_mask_attn = WhisperDecoderMaskedAttention(
            config, layer_no, safetensor_files
        )
        self.decoder_cross_attn = WhisperDecoderCrossAttention(
            config, layer_no, safetensor_files
        )
        self.decoder_ffn = WhisperDecoderFeedForwardNeuralNetwork(
            config, layer_no, safetensor_files
        )

        # layer norm
        self.mask_attn_layer_norm = torch.nn.LayerNorm(self.d_model)
        self.cross_attn_layer_norm = torch.nn.LayerNorm(self.d_model)
        self.final_layer_norm = torch.nn.LayerNorm(self.d_model)

        # weight's
        self.mask_attn_layer_norm.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.self_attn_layer_norm.weight"
            )
        )
        self.cross_attn_layer_norm.weight.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn_layer_norm.weight"
            )
        )
        self.final_layer_norm.weight.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.final_layer_norm.weight")
        )

        # bias
        self.mask_attn_layer_norm.bias.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.self_attn_layer_norm.bias"
            )
        )
        self.cross_attn_layer_norm.bias.data.copy_(
            self.get_tensor(
                f"model.decoder.layers.{layer_no}.encoder_attn_layer_norm.weight"
            )
        )
        self.final_layer_norm.bias.data.copy_(
            self.get_tensor(f"model.decoder.layers.{layer_no}.final_layer_norm.bias")
        )

    @torch.no_grad()
    def forward(self, from_decoder, from_encoder):

        residual = from_decoder

        x = self.decoder_mask_attn(from_decoder)
        x = self.mask_attn_layer_norm(x)

        x = residual + x

        residual = x

        x = self.decoder_cross_attn(x, from_encoder)
        x = self.cross_attn_layer_norm(x)

        x = residual + x

        residual = x

        x = self.decoder_ffn(x)
        x = self.final_layer_norm(x)

        x = residual + x

        return x

    def get_tensor(self, name):
        for file in self.files:
            if name in file.keys():
                return file.get_tensor(name)

In [11]:
import json

config_file_location = "config.json"

with open(config_file_location, "r") as f:
    config_file = json.load(f)

# device
device = torch.device("cpu")

# Input
dummy_input = torch.randn(1, 80, 3000).to(device)
safetensor_files = [
    safe_open(file, framework="pt", device="cpu") for file in [file_location]
]

w_encoder = WhisperEncoder(config_file, ["model.safetensors"]).to(device)
encoder_output = w_encoder(dummy_input)
encoder_output.shape

torch.Size([1, 1500, 768])

In [17]:
# Decoder
decoder_input = torch.randn(1, 8, 768)
decoder_layer = WhisperDecoderLayer(config_file, 0, safetensor_files)
decoder_layer_output = decoder_layer(decoder_input, encoder_output)

In [19]:
print(f"Input from Decoder Shape: {decoder_input.shape} | Output from encoder Shape: {encoder_output.shape}")
print("Decoder layer output:", decoder_layer_output.shape)

Input from Decoder Shape: torch.Size([1, 8, 768]) | Output from encoder Shape: torch.Size([1, 1500, 768])
Decoder layer output: torch.Size([1, 8, 768])
